# Frontier League Single-Variable Regression

This notebook runs a clean single-variable econometric analysis for one selected stat against `WinPct`. It keeps the original focused workflow while also allowing you to export a PDF report for the selected regression.


In [ ]:
from pathlib import Path
from io import StringIO
import warnings

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import jarque_bera

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["figure.dpi"] = 120

SIGNIFICANCE_LEVEL = 0.05
generated_figures = {}


## 1. Configuration and Variable Selection

Why this section exists:
- It gives you one simple selector block at the top of the notebook.
- The notebook stays focused on the core econometric question: how one selected stat relates to `WinPct`.
- The PDF export is saved in the same project folder as the notebook and CSV file.


In [ ]:
DATA_PATH = Path("frontier_league_master_stats_2021_2025_FULL.csv")

DEPENDENT_VAR = "WinPct"
INDEPENDENT_VAR = "OPS"

EXPORT_RESULTS = False
EXPORT_PDF = True
REPORT_PDF = Path("frontier_single_variable_regression_report.pdf")

ALIAS_MAP = {
    "WinPct": "Pitching_W-L%",
    "Wins": "Pitching_W",
    "Losses": "Pitching_L",
    "OPS": "Batting_OPS",
    "OBP": "Batting_OBP",
    "SLG": "Batting_SLG",
    "BA": "Batting_BA",
    "HR": "Batting_HR",
    "RBI": "Batting_RBI",
    "Runs": "Batting_R",
    "Runs_Per_Game": "Batting_R/G",
    "ERA": "Pitching_ERA",
    "RA9": "Pitching_RA9",
    "WHIP": "Pitching_WHIP",
    "FIP": "FIP",
    "H9": "Pitching_H9",
    "HR9": "Pitching_HR9",
    "BB9": "Pitching_BB9",
    "SO9": "Pitching_SO9",
    "SO_W": "Pitching_SO/W",
}


## 2. Helper Functions

Why this section exists:
- It keeps the notebook modular and easy to maintain.
- Each function handles one part of the econometric workflow: cleaning, formula building, diagnostics, interpretation, or PDF export.


In [ ]:
def resolve_variable(var_name, columns, alias_map):
    if var_name in columns:
        return var_name

    mapped_name = alias_map.get(var_name)
    if isinstance(mapped_name, str) and mapped_name in columns:
        return mapped_name

    lower_lookup = {column.lower(): column for column in columns}
    if var_name.lower() in lower_lookup:
        return lower_lookup[var_name.lower()]
    if isinstance(mapped_name, str) and mapped_name.lower() in lower_lookup:
        return lower_lookup[mapped_name.lower()]

    available_aliases = ", ".join(sorted(alias_map))
    raise KeyError(
        f"Could not resolve '{var_name}'. Use an exact CSV column name or one of these aliases: {available_aliases}"
    )


def user_label(selected_name, resolved_name):
    return selected_name if selected_name == resolved_name else f"{selected_name} -> {resolved_name}"


def formula_term_name(column_name):
    return f'Q("{column_name}")'


def render_info_text(frame):
    buffer = StringIO()
    frame.info(buf=buffer)
    return buffer.getvalue()


def load_and_clean_data(path):
    if not path.exists():
        raise FileNotFoundError(f"Could not find the data file: {path.resolve()}")

    frame = pd.read_csv(path).copy()
    original_rows = len(frame)
    frame = frame.drop_duplicates().copy()
    duplicates_removed = original_rows - len(frame)

    for column in frame.columns:
        if column == "Team":
            frame[column] = frame[column].astype("string")
        else:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame, duplicates_removed


def build_analysis_frame(frame, dependent_col, independent_col):
    keep_cols = [column for column in ["Year", "Team"] if column in frame.columns]
    keep_cols.extend([dependent_col, independent_col])
    keep_cols = list(dict.fromkeys(keep_cols))
    analysis = frame[keep_cols].copy()
    analysis = analysis.dropna(subset=[dependent_col, independent_col]).reset_index(drop=True)
    return analysis


def build_formula(dependent_col, independent_col):
    return f'{formula_term_name(dependent_col)} ~ {formula_term_name(independent_col)}'


def fit_ols(formula, analysis_frame):
    return smf.ols(formula=formula, data=analysis_frame).fit()


def make_coef_table(result):
    param_names = list(result.model.exog_names)
    conf_int = np.asarray(result.conf_int())
    table = pd.DataFrame(
        {
            "coef": np.asarray(result.params),
            "std_err": np.asarray(result.bse),
            "t_stat": np.asarray(result.tvalues),
            "p_value": np.asarray(result.pvalues),
            "ci_lower": conf_int[:, 0],
            "ci_upper": conf_int[:, 1],
        },
        index=param_names,
    )
    return table.rename_axis("term").round(4)


def make_fit_summary(result):
    return pd.Series(
        {
            "R-squared": result.rsquared,
            "Adjusted R-squared": result.rsquared_adj,
            "F-statistic": np.nan if result.fvalue is None else float(result.fvalue),
            "F-statistic p-value": np.nan if result.f_pvalue is None else float(result.f_pvalue),
            "Observations": int(result.nobs),
        }
    )


def heterosk_tests(result):
    residuals = result.resid
    exog = result.model.exog

    bp_lm, bp_lm_pvalue, bp_f, bp_f_pvalue = het_breuschpagan(residuals, exog)
    white_lm, white_lm_pvalue, white_f, white_f_pvalue = het_white(residuals, exog)

    test_table = pd.DataFrame(
        [
            {
                "Test": "Breusch-Pagan",
                "LM Statistic": bp_lm,
                "LM p-value": bp_lm_pvalue,
                "F Statistic": bp_f,
                "F p-value": bp_f_pvalue,
            },
            {
                "Test": "White",
                "LM Statistic": white_lm,
                "LM p-value": white_lm_pvalue,
                "F Statistic": white_f,
                "F p-value": white_f_pvalue,
            },
        ]
    )

    heteroskedasticity_detected = bool((test_table["LM p-value"] < SIGNIFICANCE_LEVEL).any())
    return test_table.round(4), heteroskedasticity_detected


def describe_r_squared(r_squared):
    if r_squared < 0.25:
        return "fairly weak"
    if r_squared < 0.50:
        return "modest"
    if r_squared < 0.75:
        return "fairly strong"
    return "very strong"


def interpret_simple_relation(selected_name, resolved_name, result, dependent_name, alpha=0.05):
    term_name = formula_term_name(resolved_name)
    coefficient = float(result.params[list(result.model.exog_names).index(term_name)])
    p_value = float(result.pvalues[list(result.model.exog_names).index(term_name)])

    if coefficient > 0:
        direction = f"Higher {selected_name} is associated with higher {dependent_name}"
    elif coefficient < 0:
        direction = f"Higher {selected_name} is associated with lower {dependent_name}"
    else:
        direction = f"{selected_name} does not change the fitted value of {dependent_name}"

    significance = "statistically significant" if p_value < alpha else "not statistically significant"
    magnitude = f"A one-unit increase in {selected_name} changes {dependent_name} by about {coefficient:.4f}."
    return f"{direction}. The estimated coefficient is {coefficient:.4f} and the relationship is {significance} (p = {p_value:.4f}). {magnitude}"


def create_pdf_text_figure(title, lines):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis("off")
    ax.text(0.02, 0.98, title, fontsize=18, fontweight="bold", va="top", ha="left", transform=ax.transAxes)
    ax.text(0.02, 0.92, "\n".join(lines), fontsize=11, family="monospace", va="top", ha="left", transform=ax.transAxes)
    fig.tight_layout()
    return fig


## 3. Load Data

Why this section exists:
- It validates the selected variables before the regression runs.
- It also prints the data structure, missing values, and the cleaned estimation sample.


In [ ]:
df, duplicates_removed = load_and_clean_data(DATA_PATH)
resolved_dep = resolve_variable(DEPENDENT_VAR, df.columns, ALIAS_MAP)
resolved_indep = resolve_variable(INDEPENDENT_VAR, df.columns, ALIAS_MAP)

if resolved_dep == resolved_indep:
    raise ValueError("The dependent variable cannot also be the independent variable.")

analysis_df = build_analysis_frame(df, resolved_dep, resolved_indep)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()

display(Markdown("### Resolved Variable Selection"))
print(f"Dependent variable: {user_label(DEPENDENT_VAR, resolved_dep)}")
print(f"Independent variable: {user_label(INDEPENDENT_VAR, resolved_indep)}")
print(f"\nRows after duplicate removal: {len(df)}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Rows available for estimation after dropping missing model values: {len(analysis_df)}")

display(Markdown("### DataFrame Info"))
print(render_info_text(df))

display(Markdown("### Missing Value Summary"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

display(Markdown("### Numeric Columns"))
print(numeric_columns)

display(Markdown("### Preview of the Analysis Sample"))
display(analysis_df.head())


## 4. Standard Single-Variable OLS Regression

Why this section exists:
- Simple regression estimates the average bivariate relationship between the chosen statistic and winning percentage.
- The coefficient sign gives direction, the p-value gives statistical evidence, and R-squared shows explanatory power.

How to read the regression output:
- `coef`: the estimated change in `WinPct` from a one-unit change in the selected stat.
- `std_err`: the standard error of the coefficient estimate. This tells you how much uncertainty surrounds the estimate.
- `t_stat`: the coefficient divided by its standard error. Larger absolute values usually indicate stronger evidence of a non-zero relationship.
- `p_value`: the probability of seeing a result this extreme if the true coefficient were zero. Values below 0.05 are often treated as statistically significant.
- `ci_lower` and `ci_upper`: the lower and upper confidence interval bounds for the estimate.
- `R-squared`: the share of variation in `WinPct` explained by the selected stat.
- `Adjusted R-squared`: a slightly penalized version of R-squared.
- `F-statistic`: a test of whether the selected regressor helps explain `WinPct`.


In [ ]:
formula = build_formula(resolved_dep, resolved_indep)
model = fit_ols(formula, analysis_df)
term_label_map = {formula_term_name(resolved_indep): INDEPENDENT_VAR}

coef_table = make_coef_table(model).rename(index=term_label_map)
fit_summary = make_fit_summary(model).round(4)

display(Markdown(f"### Model Formula\n`{formula}`"))
display(Markdown("### Coefficient Table"))
display(coef_table)

display(Markdown("### Overall Model Fit"))
display(fit_summary.to_frame("value"))
display(
    Markdown(
        """### How To Read These Results
- `coef`: estimated change in `WinPct` from a one-unit increase in the selected stat.
- `std_err`: the uncertainty around that coefficient estimate. Smaller usually means more precision.
- `t_stat`: coefficient divided by its standard error. Bigger absolute values usually mean stronger evidence.
- `p_value`: if this is small, often below 0.05, the relationship is commonly treated as statistically significant.
- `ci_lower` and `ci_upper`: the confidence interval for the coefficient. If zero is outside the interval, that usually matches significance.
- `R-squared`: the share of variation in `WinPct` explained by the selected stat.
- `Adjusted R-squared`: a slightly penalized version of `R-squared`.
- `F-statistic`: tests whether the regression is useful overall.
"""
    )
)


## 5. Plots and Heteroskedasticity Tests

Why this section exists:
- The scatterplot and regression line show the fitted relationship visually.
- The residual plot helps show whether the model misses patterns.
- Breusch-Pagan and White tests check whether the error variance is constant; if not, robust standard errors are safer for inference.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.regplot(
    data=analysis_df,
    x=resolved_indep,
    y=resolved_dep,
    ci=95,
    scatter_kws={"s": 90, "alpha": 0.85},
    line_kws={"color": "crimson", "linewidth": 3},
    ax=axes[0],
)
axes[0].set_title("Scatterplot with OLS Regression Line")
axes[0].set_xlabel(INDEPENDENT_VAR)
axes[0].set_ylabel(DEPENDENT_VAR)

sns.scatterplot(x=model.fittedvalues, y=model.resid, s=90, ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Fitted Values")
axes[1].set_ylabel("Residuals")

fig.tight_layout()
generated_figures["regression_and_residual_plot"] = fig
plt.show()

heterosk_table, heteroskedasticity_detected = heterosk_tests(model)
display(Markdown("### Heteroskedasticity Tests"))
display(heterosk_table)
print("Null hypothesis: the regression errors have constant variance (homoskedasticity).")
print("Alternative hypothesis: the regression errors have non-constant variance (heteroskedasticity).")

if heteroskedasticity_detected:
    robust_model = model.get_robustcov_results(cov_type="HC1")
    robust_coef_table = make_coef_table(robust_model).rename(index=term_label_map)
    print("At least one test rejects the null at the 5% level, so HC1 robust standard errors are reported below.")
    display(Markdown("### HC1 Robust Coefficient Table"))
    display(robust_coef_table)
else:
    robust_model = None
    robust_coef_table = None
    print("Neither test rejects the null at the 5% level, so the classical OLS standard errors are retained.")


## 6. Residual Diagnostics, Interpretation, and PDF Export

Why this section exists:
- Residual checks help judge whether the linear-model assumptions look credible.
- The automatic interpretation turns the output into readable econometric language.
- The PDF export creates a clean report for the selected regression and saves it in this project folder.


In [ ]:
interpretation_lines = []

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(model.resid, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Histogram of Residuals")
axes[0].set_xlabel("Residual")

sm.qqplot(model.resid, line="45", fit=True, ax=axes[1])
axes[1].set_title("QQ Plot of Residuals")

fig.tight_layout()
generated_figures["residual_distribution"] = fig
plt.show()

jb_stat, jb_pvalue, jb_skew, jb_kurtosis = jarque_bera(model.resid)
print("Jarque-Bera normality test")
print("Null hypothesis: the residuals follow a normal distribution.")
print("Alternative hypothesis: the residuals are not normally distributed.")
print(f"JB statistic: {jb_stat:.4f}")
print(f"JB p-value: {jb_pvalue:.4f}")
print(f"Residual skewness: {jb_skew:.4f}")
print(f"Residual kurtosis: {jb_kurtosis:.4f}")
if jb_pvalue < SIGNIFICANCE_LEVEL:
    print("Interpretation: reject normality at the 5% level, so normal-theory inference should be treated cautiously.")
else:
    print("Interpretation: do not reject normality at the 5% level, so the residual distribution does not show a strong normality violation.")

reporting_model = robust_model if robust_model is not None else model
interpretation_lines = [
    f"- {interpret_simple_relation(INDEPENDENT_VAR, resolved_indep, reporting_model, DEPENDENT_VAR, SIGNIFICANCE_LEVEL)}",
    f"- The model R-squared is {model.rsquared:.3f}, so the selected regressor explains about {model.rsquared:.1%} of the variation in {DEPENDENT_VAR}. This is {describe_r_squared(model.rsquared)} explanatory power.",
]

if robust_model is not None:
    interpretation_lines.append(
        "- HC1 robust standard errors were used for inference because the heteroskedasticity tests suggested non-constant error variance."
    )

display(Markdown("### Automatic Interpretation"))
display(Markdown("\n".join(interpretation_lines)))

if EXPORT_PDF:
    with PdfPages(REPORT_PDF) as pdf:
        cover_lines = [
            f"Dependent variable: {user_label(DEPENDENT_VAR, resolved_dep)}",
            f"Independent variable: {user_label(INDEPENDENT_VAR, resolved_indep)}",
            f"Rows used: {len(analysis_df)}",
            f"PDF location: {REPORT_PDF.resolve()}",
        ]
        cover_fig = create_pdf_text_figure("Frontier League Single-Variable Regression Report", cover_lines)
        pdf.savefig(cover_fig, bbox_inches="tight")
        plt.close(cover_fig)

        standard_lines = [
            f"Formula: {formula}",
            "",
            "Coefficient Table",
            coef_table.to_string(),
            "",
            "Overall Model Fit",
            fit_summary.to_frame("value").round(4).to_string(),
            "",
            "Heteroskedasticity Tests",
            heterosk_table.to_string(index=False) if heterosk_table is not None else "Not run",
            "",
            "Interpretation",
            *interpretation_lines,
            "",
            "Jarque-Bera",
            f"JB statistic = {jb_stat:.4f}, p-value = {jb_pvalue:.4f}, skewness = {jb_skew:.4f}, kurtosis = {jb_kurtosis:.4f}",
        ]
        summary_fig = create_pdf_text_figure("Standard Regression Summary", standard_lines)
        pdf.savefig(summary_fig, bbox_inches="tight")
        plt.close(summary_fig)

        for figure_name, figure in generated_figures.items():
            pdf.savefig(figure, bbox_inches="tight")

    print(f"PDF report saved to: {REPORT_PDF.resolve()}")
else:
    print("EXPORT_PDF = False, so no PDF was written.")
